# SemanticEmbed: Drift Detection

Compare two versions of a graph and see exactly what structural properties changed, per node, per dimension.

[GitHub](https://github.com/jmurray10/semanticembed-sdk)


In [ ]:
!pip install -q semanticembed
from semanticembed import encode, report, drift
print("Ready.")


---
## The Scenario

You have a microservice architecture. A developer submits a PR that adds a new dependency. Before merging, you want to know: what structural risks did this change introduce?


In [ ]:
# Before: Sock Shop baseline
edges_before = [
    ("edge-router",   "frontend"),
    ("frontend",      "catalogue"),
    ("frontend",      "orders"),
    ("frontend",      "user"),
    ("frontend",      "carts"),
    ("orders",        "shipping"),
    ("orders",        "payment"),
    ("orders",        "carts"),
    ("orders",        "user"),
    ("catalogue",     "catalogue-db"),
    ("carts",         "carts-db"),
    ("orders",        "orders-db"),
    ("user",          "user-db"),
    ("shipping",      "rabbitmq"),
    ("queue-master",  "rabbitmq"),
]

# After: Someone adds a direct frontend -> payment edge
edges_after = edges_before + [("frontend", "payment")]

print(f"Before: {len(edges_before)} edges")
print(f"After:  {len(edges_after)} edges")
print(f"Change: added frontend -> payment")


---
## Compute Both Encodings


In [ ]:
result_before = encode(edges_before)
result_after = encode(edges_after)

print("BEFORE:")
print(result_before.table)
print()
print("AFTER:")
print(result_after.table)


---
## Drift Report

One function call shows exactly what changed.


In [ ]:
changes = drift(result_before, result_after)

print("STRUCTURAL DRIFT REPORT")
print("=" * 60)
print("Change: added edge frontend -> payment")
print()

if not changes:
    print("No structural changes detected.")
else:
    for node, deltas in sorted(changes.items()):
        print(f"{node}:")
        for dim, delta in deltas.items():
            direction = "+" if delta > 0 else ""
            print(f"  {dim:<15} {direction}{delta:.4f}")
        print()


---
## Risk Comparison

Compare risk reports before and after the change.


In [ ]:
risk_before = report(result_before)
risk_after = report(result_after)

print("RISKS BEFORE:")
print(risk_before)
print()
print("RISKS AFTER:")
print(risk_after)


---
## CI/CD Integration

In production, this runs as a CI/CD gate:

```yaml
# .github/workflows/structural-check.yml
- name: Structural risk check
  run: |
    semanticembed diff topology-before.json topology-after.json \
      --fail-on critical \
      --report pr-comment
```

A PR that introduces a new structural SPOF gets flagged before merge. A PR that adds a safe leaf service passes automatically.

---

**Next:** [04 - Bring Your Own Graph](https://colab.research.google.com/github/jmurray10/semanticembed-sdk/blob/main/notebooks/04_bring_your_own.ipynb)

*Patent pending. Application #63/994,075.*


## Async drift: encode both versions in parallel

For larger graphs the two encode calls can run concurrently:

```python
import asyncio, semanticembed as se

changes = asyncio.run(se.aencode_diff(edges_v1, edges_v2))
for node, deltas in changes.items():
    for dim, info in deltas.items():
        print(f"{node}.{dim}: {info['before']:.3f} -> {info['after']:.3f}")
```

`aencode_diff` issues both encodes via `asyncio.gather`, cuts wall time roughly
in half on 2+ second cold starts.
